# Pesquisa Avançada: Detecção de Câncer via Feature Engineering Progressiva
**Objetivo:** Medir o ganho incremental de Cor, Textura e Espectro na classificação de lesões de pele.

---

In [ ]:
!pip install datasets tensorflow pandas matplotlib seaborn scikit-learn scikit-image -q
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from datasets import load_dataset
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.color import rgb2lab, rgb2hsv
from skimage.measure import moments, moments_hu
from skimage.filters import threshold_otsu
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
import gc

print("--- Configurando Acelerador ---")
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local')
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    print('Hardware: TPU Ativa!')
except:
    gpus = tf.config.list_physical_devices('GPU')
    if len(gpus) > 1:
        strategy = tf.distribute.MirroredStrategy()
        print(f'Hardware: {len(gpus)} GPUs Ativas (MirroredStrategy)')
    elif len(gpus) == 1:
        strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
        print('Hardware: 1 GPU Ativa')
    else:
        strategy = tf.distribute.get_strategy()
        print('Hardware: CPU')

IMG_SIZE = (224, 224)
BATCH_SIZE = 16 * strategy.num_replicas_in_sync

## 1. Carregamento e Balanceamento (Máx 4k Nevi)
Usamos streaming para garantir que o download não trave no Kaggle.

In [ ]:
print("--- Iniciando Streaming Híbrido ---")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except: hf_token = None

ds_stream = load_dataset("marmal88/skin_cancer", split='train', streaming=True, token=hf_token)

limit_nevi = 4000
counts = {}
X_raw, y_labels = [], []

for example in ds_stream:
    lbl = example['dx']
    counts[lbl] = counts.get(lbl, 0)
    
    # Se for Nevi e já atingiu 4000, pula. Senão, pega tudo.
    if lbl == 'melanocytic_Nevi' and counts[lbl] >= limit_nevi:
        continue
        
    img = example['image'].convert('RGB').resize(IMG_SIZE)
    X_raw.append(np.array(img, dtype='uint8'))
    y_labels.append(lbl)
    counts[lbl] += 1
    if sum(counts.values()) % 1000 == 0: print(f"Capturadas: {sum(counts.values())} | {counts}")

X_raw = np.array(X_raw)
y_labels = np.array(y_labels)
gc.collect()
print(f"Dataset Finalizado: {X_raw.shape}")

## 2. Feature Engineering Progressiva
Criamos extratores para Cor, Textura e Espectro.

In [ ]:
from PIL import Image
print("--- Extração Avançada: DullRazor + ABCD (GLCM+LBP) ---")

def dull_razor(img_uint8):
    # 1. Grayscale
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    # 2. Black Hat Transform (destaca estruturas escuras e finas como pelos)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    # 3. Threshold para máscara de pelos
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    # 4. Inpainting (preenchimento dos pelos com pixels vizinhos)
    dst = cv2.inpaint(img_uint8, mask, 1, cv2.INPAINT_TELEA)
    return dst

def extract_all_features(img_raw):
    # Pré-processamento: Remoção de pelos
    img_uint8 = dull_razor(img_raw)
    
    img_f = img_uint8.astype('float32') / 255.0
    img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    
    # 1. COR (RGB + HSV) - Regra 'C'
    color_rgb = img_f.mean(axis=(0,1))
    img_hsv = rgb2hsv(img_f)
    color_hsv = img_hsv.mean(axis=(0,1))
    color_feat = np.concatenate([color_rgb, color_hsv])
    
    # 2. TEXTURA (GLCM + LBP) - Regra 'B'
    # GLCM (Macro-textura)
    glcm = graycomatrix(img_gray, [1], [0], levels=256, symmetric=True, normed=True)
    texture_glcm = [
        graycoprops(glcm, 'contrast')[0,0], 
        graycoprops(glcm, 'homogeneity')[0,0],
        graycoprops(glcm, 'energy')[0,0],
        graycoprops(glcm, 'correlation')[0,0]
    ]
    # LBP (Micro-textura - padrões de pontos/redes)
    lbp = local_binary_pattern(img_gray, 8, 1, method='uniform')
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 10 + 3), range=(0, 10 + 2))
    texture_lbp = hist.astype("float")
    texture_lbp /= (texture_lbp.sum() + 1e-7)
    texture_feat = np.concatenate([texture_glcm, texture_lbp])
    
    # 3. FORMA (Hu Moments) - Regra 'A'
    try:
        thresh = threshold_otsu(img_gray)
        binary = (img_gray > thresh).astype(np.uint8)
        if binary[0,0] == 1: binary = 1 - binary
        mu = moments(binary)
        hu = moments_hu(mu)
        hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    except:
        hu = np.zeros(7)
    shape_feat = hu
    
    return color_feat, texture_feat, shape_feat

print("Processando imagens com DullRazor...")
feats = [extract_all_features(X_raw[i]) for i in range(len(X_raw))]
f_color = np.nan_to_num(np.array([f[0] for f in feats]))
f_texture = np.nan_to_num(np.array([f[1] for f in feats]))
f_shape = np.nan_to_num(np.array([f[2] for f in feats]))

X_final = np.hstack([f_color, f_texture, f_shape])
print(f"Finalizado! Atributos totais: {X_final.shape[1]}")

## 3. Análise Diagnóstica (EDA & PCA Visual)
Antes do treino, analisamos a separabilidade das classes e a redundância dos atributos.

In [ ]:
print("--- Gerando Insights Diagnósticos ---")

# 1. Mapa de Correlação (EDA)
df_feats = pd.DataFrame(X_final)
plt.figure(figsize=(12, 10))
sns.heatmap(df_feats.corr(), cmap='RdBu_r', center=0)
plt.title("Matriz de Correlação entre Atributos ABCD")
plt.show()

# 2. Visualização de Separação de Classes (PCA 2D)
pca_vis = PCA(n_components=2, random_state=42)
X_vis = pca_vis.fit_transform(StandardScaler().fit_transform(X_final))

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_vis[:, 0], X_vis[:, 1], c=y_enc, cmap='viridis', alpha=0.6, s=30)
plt.colorbar(scatter, ticks=range(len(le.classes_)), label='Classes')
plt.title("Visualização 2D do Espaço de Atributos (PCA)")
plt.xlabel(f"PC1 ({pca_vis.explained_variance_ratio_[0]:.1%} da var)")
plt.ylabel(f"PC2 ({pca_vis.explained_variance_ratio_[1]:.1%} da var)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

print(f"Variância total explicada pelos 2 componentes: {sum(pca_vis.explained_variance_ratio_):.1%}")

## 4. Gráfico de Ganho (Random Forest)
Medindo a importância de cada técnica.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_enc = le.fit_transform(y_labels)

results = []
def test_rf(X_data, name):
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
    
    # Balanceamento via Oversampling para garantir que classes minoritárias sejam ouvidas
    ros = RandomOverSampler(random_state=42)
    X_resampled, y_resampled = ros.fit_resample(X_train, y_train)
    
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X_resampled, y_resampled)
    acc = accuracy_score(y_test, rf.predict(X_test))
    results.append({'Tecnica': name, 'Acuracia': acc})
    return rf

print("Treinando modelos incrementais...")
test_rf(f_color, "A: Cor (HSV)")
test_rf(np.hstack([f_color, f_texture]), "B: Cor + Tex (GLCM+LBP)")
rf_final = test_rf(X_final, "C: Final (ABCD + DullRazor)")

df_res = pd.DataFrame(results)
plt.figure(figsize=(10,5))
sns.lineplot(data=df_res, x='Tecnica', y='Acuracia', marker='o', linewidth=3)
plt.title("Gráfico de Ganho por Técnica (Random Forest)")
plt.grid(True)
plt.show()

## 5. Deep Learning: Viabilidade de Alta Fidelidade
Modelo Dual-Input para validar o ganho com redes neurais.

In [ ]:
def dl_pipeline(img_uint8, feats, label):
    def process(img):
        img_f = img.astype('float32') / 255.0
        return img_f
    
    rgb = tf.numpy_function(process, [img_uint8], tf.float32)
    rgb.set_shape((224, 224, 3))
    return {"rgb_input": rgb, "abcd_input": feats}, label

# Normalização dos atributos ABCD para a rede neural
scaler_dl = StandardScaler()
X_final_scaled = scaler_dl.fit_transform(X_final)

X_tr, X_ts, y_tr, y_ts, f_tr, f_ts = train_test_split(X_raw, y_enc, X_final_scaled, test_size=0.15, stratify=y_enc)

train_ds = tf.data.Dataset.from_tensor_slices((X_tr, f_tr, y_tr)).shuffle(1000).map(dl_pipeline).batch(BATCH_SIZE).prefetch(2)
test_ds = tf.data.Dataset.from_tensor_slices((X_ts, f_ts, y_ts)).map(dl_pipeline).batch(BATCH_SIZE).prefetch(2)

# Pesos de classe agressivos para Melanoma (classe 5) e Carcinomas
weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
class_weight_dict = dict(enumerate(weights))
class_weight_dict[5] *= 1.5 # Reforço extra para Melanoma
class_weight_dict[1] *= 1.2 # Reforço para Basal Cell

with strategy.scope():
    data_aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.1),
    ])

    # Ramo 1: Visão (EfficientNet)
    in_rgb = layers.Input(shape=(224, 224, 3), name="rgb_input")
    x1 = data_aug(in_rgb)
    x1 = tf.keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet')(x1)
    x1 = layers.GlobalAveragePooling2D()(x1)
    x1 = layers.Dense(128, activation='relu')(x1)
    
    # Ramo 2: Atributos Clínicos (ABCD)
    in_abcd = layers.Input(shape=(X_final.shape[1],), name="abcd_input")
    x2 = layers.Dense(32, activation='relu')(in_abcd)
    
    # Fusão de Características
    comb = layers.Concatenate()([x1, x2])
    comb = layers.Dropout(0.3)(comb)
    out = layers.Dense(len(le.classes_), activation='softmax')(comb)
    
    model = models.Model(inputs=[in_rgb, in_abcd], outputs=out)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-4), 
        loss='sparse_categorical_crossentropy', 
        metrics=['accuracy']
    )

print("Treinando Modelo Híbrido (Vision + ABCD)...")
model.fit(train_ds, validation_data=test_ds, epochs=25, class_weight=class_weight_dict)

y_pred = np.argmax(model.predict(test_ds), axis=1)
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_ts, y_pred), annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.title("Matriz de Confusão Final (Deep Learning)")
plt.show()